# Reproducing Prince et al. (2025) from dandiset 001371

Paper. Prince, Cushing, Yassine, Katragadda, Roberts & Singer (2025), "New information triggers prospective codes to adapt for flexible navigation", *Nat. Commun.* 16, 4822, doi:10.1038/s41467-025-60122-8.

Dataset. DANDI:001371, version `0.260831.0201` (doi:10.48324/dandi.001371/0.260831.0201), streamed from the archive. The paper cites the dandiset without a version; this is the only published version (the draft has the same 66 assets), so it is pinned.

Claims. `claims/001371/10.1038-s41467-025-60122-8.yaml`, written from the version of record (Europe PMC, PMC12102184) with its Supplementary Information, and the paper's code (github.com/stephprince/update-project at d8557f0). The shared method S1 to S11 of that file is cited below as "S1" and so on.

| Claim | Tested here |
|---|---|
| recordings, trial-mix, task-timing, fig1b-above-chance, fig1b-window-performance, fig1d-heading-switch, fig5a-stay-performance | tested (behaviour and counts) |
| tuning-tiles-track, decoding-validation | tested (position decoding) |
| fig2e-hpc-goal-increase, fig2e-hpc-arms-equal, sfig2e-goal-bias, sfig3c-goals-over-retrospective, sfig3c-retrospective-unchanged, sfig3ab-original-vs-update-cue, sfig2f-all-trials-encoder | tested (CA1 position decoding) |
| sfig5f-pfc-wider-spread, sfig5g-ca1-more-goal-coding, fig2g-pfc-no-goal-increase, sfig5e-pfc-windows | tested (PFC and CA1 position decoding) |
| fig3c-theta-phase-segregation, fig3b-theta-pre-post, sfig6-theta-switch-over-delay, fig3d-new-arm-theta-modulated | tested (CA1 decoding in 25 ms bins with theta phase) |
| fig5d-stay-hpc-like-delay, fig5d-stay-hpc-below-switch, sfig10-stay-transient-new-arm | tested |
| fig6b-hpc-new-lower-incorrect, fig6b-hpc-initial-no-difference, fig6-hpc-pre-cue-initial-higher-incorrect, sfig12-delay-hpc-no-outcome-difference | tested (residual GLM) |
| fig7c-hpc-commitment-new-arm, fig7c-hpc-commitment-initial-arm | tested |
| fig4a-lstm-choice-estimate and every claim on choice decoding (fig4f, fig4d, sfig7d, sfig7e, fig4h, fig4g, fig5f, fig6d, sfig12-delay-pfc, fig6e, fig7e) | blocked, `too_large`: the per-session LSTM is timed on one session below and the whole does not fit |

The notebook takes about 25 minutes, most of it streaming the 66 files (only units, trials, behaviour and one LFP channel are read).

In [ ]:
import os, time, warnings
os.environ.setdefault("OMP_NUM_THREADS", "1")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import concurrent.futures as cf
import prince2025 as P
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("dandiset", P.DANDISET, "version", P.VERSION)

In [ ]:
# Stream every session and reduce it to what the analyses use (prince2025.process_session).
ASSETS = P.list_assets()
t0 = time.time()
def _job(i):
    a = ASSETS.iloc[i]
    return P.process_session(a.asset_id, a.subject, a.session, seed=i)   # each session its own seed
with cf.ProcessPoolExecutor(8) as ex:
    R = list(ex.map(_job, range(len(ASSETS))))
print(f"{len(R)} sessions processed in {time.time() - t0:.0f} s")

In [ ]:
# ---- Data check -------------------------------------------------------------------------
print("assets:", len(ASSETS), "subjects:", ASSETS.subject.nunique(), ASSETS.groupby("subject").size().to_dict())
print("asset sizes (GB): %.0f to %.0f" % (ASSETS.size_gb.min(), ASSETS.size_gb.max()))
S = pd.DataFrame([dict(subject=r["subject"], session=r["session"], CA1=r["n_units"]["CA1"], PFC=r["n_units"]["PFC"],
                       no_region=r["n_units"].get("unknown", 0), trials_all=len(r["trials_all"]),
                       final=len(r["trials"]), warmup=int((r["trials_all"].maze_id == 3).sum()),
                       other_maze=int((~r["trials_all"].maze_id.isin([3, 4])).sum()),
                       delay=int((r["trials"].update_type == 1).sum()), switch=int((r["trials"].update_type == 2).sum()),
                       stay=int((r["trials"].update_type == 3).sum()), n_train=r["n_train"],
                       quality=str(r["quality"])) for r in R])
print(S.to_string())
print("\ntotals:", S[["CA1", "PFC", "no_region", "trials_all", "final", "warmup", "delay", "switch", "stay"]].sum().to_dict())
print("unit quality labels over all sessions:", set(S.quality))
ct = pd.concat([r["cell_types"] for r in R]).groupby(level=0).sum()
print("\ncell types by region:\n", ct)
nore = S[S.no_region > 0]
print("\nsessions whose units table has no region or cell type columns:", list(nore.session), "units:", int(nore.no_region.sum()))
# can the region of these units be recovered? what their units table holds, and whether the phy channel separates the probes
for r in R:
    if r["n_units"]["unknown"]:
        print(f"  {r['session']}: units columns {r['unit_cols']}; electrodes {r['electrode_locations']}; phy channel 'ch' range {r['ch_range']['unknown']}")
lab = [r for r in R if not r["n_units"]["unknown"]]
print("  labelled sessions, phy channel 'ch' range by region (over sessions): CA1 %.0f to %.0f, PFC %.0f to %.0f" % (
    min(r["ch_range"]["CA1"][0] for r in lab if "CA1" in r["ch_range"]), max(r["ch_range"]["CA1"][1] for r in lab if "CA1" in r["ch_range"]),
    min(r["ch_range"]["PFC"][0] for r in lab if "PFC" in r["ch_range"]), max(r["ch_range"]["PFC"][1] for r in lab if "PFC" in r["ch_range"])))
print("  units tables with an 'electrodes' or 'electrode_group' column:", sum(("electrodes" in r["unit_cols"]) or ("electrode_group" in r["unit_cols"]) for r in R),
      "of", len(R), "; of the unlabelled sessions:", sum(("electrodes" in r["unit_cols"]) or ("electrode_group" in r["unit_cols"]) for r in R if r["n_units"]["unknown"]))

In [ ]:
# Units and ranges of every variable used, and the metadata stated in the files.
ua = R[0]["units_attrs"]
print("metadata as stated:", ua, "(identical in all sessions:", all(r["units_attrs"] == ua for r in R), ")")
pos_lo = np.min([r["pos_range"][0] for r in R], 0); pos_hi = np.max([r["pos_range"][1] for r in R], 0)
print("position x range %.1f to %.1f, y range %.1f to %.1f (stated unit 'meters')" % (pos_lo[0], pos_hi[0], pos_lo[1], pos_hi[1]))
va = np.array([r["va_range"] for r in R]); print("view angle range %.3f to %.3f (stated unit 'degrees')" % (va[:, 0].min(), va[:, 1].max()))
lin = np.array([r["lin_range"] for r in R]); print("linearized position range %.1f to %.1f" % (lin[:, 0].min(), lin[:, 1].max()))
print("moving fraction of behaviour samples (|rot|+|trans| > 1000): %.3f to %.3f" % (min(r["moving_frac"] for r in R), max(r["moving_frac"] for r in R)))
h = np.sum([r["acq_speed_hist"] for r in R], 0); e = np.linspace(0, 5, 51)
print("log10(|rot|+|trans|+1) histogram of the acquisition series, all sessions (bin start: count):")
print({round(float(a), 1): int(c) for a, c in zip(e[:-1], h)})
# clocks: spikes, behaviour and trials within the same span
bad = [r["session"] for r in R if not (r["spike_range"][0] < r["trials"].start_time.min() and r["spike_range"][1] > r["trials"].stop_time.max() * 0.9)]
print("sessions whose spikes do not span the trials:", bad)

In [ ]:
# Trial-table checks: the meaning of turn_type, of the cue times, and of view angle sign.
rows = []
for r in R:
    tr = r["trials"]
    for _, t in tr.iterrows():
        rows.append(dict(session=r["session"], update_type=t.update_type, correct=t.correct, turn_type=t.turn_type,
                         choice=t.choice))
TT = pd.DataFrame(rows)
print("correct trials: final choice by turn_type and trial type")
print(pd.crosstab([TT[TT.correct == 1].update_type, TT[TT.correct == 1].turn_type], TT[TT.correct == 1].choice))
g = R[0]["geo"]
print("\nlinearized segments: arm bins left", np.where(g["left"])[0], "right", np.where(g["right"])[0])

In [ ]:
# Checks of the trial table against the behaviour (continued).
ALLT = pd.concat([r["trials"].assign(animal=r["subject"], session=r["session"]) for r in R], ignore_index=True)
c = ALLT[ALLT.correct == 1]
print("correct trials, sign of x at choice made by turn_type and trial type:")
print(pd.crosstab([c.update_type, c.turn_type], np.sign(c.x_at_choice)))
print("\ny position at t_update on switch and stay trials: median %.2f, 1st to 99th percentile %.2f to %.2f (update cue start 145.125 in the code)"
      % tuple(np.nanpercentile(ALLT[ALLT.update_type > 1].y_at_update, [50, 1, 99])))
print("y position at t_delay: median %.2f, 1st to 99th percentile %.2f to %.2f (delay cue start 120.325 in the code)"
      % tuple(np.nanpercentile(ALLT.y_at_delay, [50, 1, 99])))
print("y at the stored update_location minus y at t_update: median %.3f" % np.nanmedian(ALLT.update_location - ALLT.y_at_update))
r_va = np.array([r["va_dx_r"] for r in R])
print("\nSpearman r between view angle and sideways movement dx in the central arm, per session: median %.2f, range %.2f to %.2f"
      % (np.median(r_va), r_va.min(), r_va.max()))

In [ ]:
# Clock of spikes against behaviour: position decoding error on held-out delay only trials with the
# position at the same time, and with the position 2 s earlier or later (per region, included sessions).
for reg, rule in (("CA1", "final"), ("PFC", "final"), ("CA1", "train"), ("PFC", "train")):
    rows = [dict(session=r["session"], same=np.median(r["dec"][reg]["val_err"]), minus2=r["dec"][reg].get("val_err_shift", {}).get(-2.0, np.nan),
                 plus2=r["dec"][reg].get("val_err_shift", {}).get(2.0, np.nan)) for r in R if P.included(r, reg, rule)]
    d = pd.DataFrame(rows)
    print(f"{reg} rule={rule}: {len(d)} sessions; median error (linearized units) same time {d.same.median():.1f}, "
          f"-2 s {d.minus2.median():.1f}, +2 s {d.plus2.median():.1f}; sessions where same time is lowest: "
          f"{int(((d.same < d.minus2) & (d.same < d.plus2)).sum())} of {len(d)}")

## recordings

> Seven mice were recorded over 66 sessions, yielding 3892 hippocampal CA1 units and 2557 medial prefrontal units after curation and cell type classification.

In [ ]:
n_animals = ASSETS.subject.nunique(); n_sessions = len(ASSETS)
nCA1, nPFC, nNo = S.CA1.sum(), S.PFC.sum(), S.no_region.sum()
print("animals", n_animals, "(expected 7); sessions", n_sessions, "(expected 66)")
print("CA1 units", nCA1, "(expected 3892 +/- 389.2); PFC units", nPFC, "(expected 2557 +/- 255.7)")
print("units without region:", nNo, "; all units", nCA1 + nPFC + nNo, "against the paper's total", 3892 + 2557)
print("\nper animal (Supplementary Table 1 gives per-animal counts beside the totals):")
print(S.groupby("subject")[["CA1", "PFC", "no_region"]].sum().assign(sessions=S.groupby("subject").size()))
print("\nwithin tolerance: CA1", abs(nCA1 - 3892) <= 389.2, "PFC", abs(nPFC - 2557) <= 255.7)

**Result: partially reproduced.** 7 animals and 66 sessions match, with 8, 12, 10, 11, 12, 6 and 7 sessions per animal. The CA1 count (3545) is within tolerance and the PFC count (2271) is not. The units of five sessions carry no region or cell type; with them the three counts sum exactly to the paper's 6449 units (3545 + 2271 + 633), which is consistent with the shortfall being those unlabelled units; how the 633 units divide between the two regions is not known. Their region cannot be recovered from the files: their units tables have no `electrodes` or `electrode_group` column (0 of the 5; 61 of 66 tables over all sessions have one), and the phy channel `ch` runs from 0 to 63 on both probes in the labelled sessions (CA1 0 to 63, PFC 0 to 63), so it does not say which probe a unit is on (data check).

## trial-mix

> In the final task 65% of trials are delay only, 25% switch and 10% stay trials.

In [ ]:
F = ALLT  # final-task trials (maze_id 4)
pooled = F.update_type.value_counts(normalize=True).sort_index()
per = F.groupby("session").update_type.value_counts(normalize=True).unstack().fillna(0).mean()
exp = {1: (0.65, 0.065), 2: (0.25, 0.025), 3: (0.10, 0.01)}
for name, fr in (("(a) pooled", pooled), ("(b) mean of sessions", per)):
    ok = all(abs(fr[k] - v) <= tol for k, (v, tol) in exp.items())
    print(name, {P.TRIAL_TYPES[k]: round(float(fr[k]), 4) for k in exp}, "all within tolerance:", ok)
print("counts:", F.update_type.map(P.TRIAL_TYPES).value_counts().to_dict(), "of", len(F))

**Result: not reproduced.** The realized fractions are 0.737, 0.189 and 0.074 (delay only, switch, stay) pooled over the 9729 final-task trials and 0.750, 0.181 and 0.069 averaged over sessions, outside the tolerances under both readings, as the claims file foresaw from the paper's own counts (Supplementary Table 1). The 65/25/10% read as programmed probabilities, which the realized trials do not show. The 932 warm-up trials (maze_id 3) and the 7172 final-task delay only trials make 8104, the delay count of Supplementary Table 1 (data check).

## task-timing

> The delay on delay only trials lasted 10.56 s on average, the shortened delay before the update cue on switch trials 1.40 s, and the delay after the update cue 6.02 s; trials lasted about 21 s from start to choice.

In [ ]:
def msem(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return x.mean(), x.std(ddof=1) / np.sqrt(len(x)), len(x)
rows = []
for tset, sub in (("(c) all", F), ("(d) correct", F[F.correct == 1])):
    d1 = sub[sub.update_type == 1]; sw = sub[sub.update_type == 2]; ss = sub[sub.update_type.isin([2, 3])]
    vals = {"delay, delay only (t_choice_made - t_delay)": (d1.t_choice_made - d1.t_delay, 10.56, 0.14),
            "first delay, switch (t_update - t_delay)": (sw.t_update - sw.t_delay, 1.40, 0.06),
            "second delay (a) t_choice_made - t_delay2, (e) switch": (sw.t_choice_made - sw.t_delay2, 6.02, 0.16),
            "second delay (b) t_choice_made - t_update, (e) switch": (sw.t_choice_made - sw.t_update, 6.02, 0.16),
            "second delay (a), (f) switch and stay": (ss.t_choice_made - ss.t_delay2, 6.02, 0.16),
            "second delay (b), (f) switch and stay": (ss.t_choice_made - ss.t_update, 6.02, 0.16),
            "trial start to choice, delay only": (d1.t_choice_made - d1.start_time, 20.93, 0.18),
            "trial start to choice, switch": (sw.t_choice_made - sw.start_time, 20.79, 0.45)}
    for k, (x, e, tol) in vals.items():
        m, se, n = msem(x)
        rows.append(dict(trials=tset, quantity=k, mean=round(m, 3), sem=round(se, 3), n=n, expected=e, within=abs(m - e) <= tol))
TIM = pd.DataFrame(rows); print(TIM.to_string())

**Result: reproduced.** Over all final-task trials, the delay on delay only trials is 10.559 +/- 0.067 s (n = 7172), the first delay on switch trials 1.403 +/- 0.029 s (n = 1840), the second delay from its onset (t_delay2) to choice made 6.020 +/- 0.084 s on switch trials, and trial start to choice 21.00 s (delay only) and 20.88 s (switch), all within tolerance (readings a, c, e; the trial counts equal the paper's). Reading (b), update cue onset to choice, gives 9.93 s and does not match. On correct trials only, the delay only delay (10.41 s) and first delay (1.33 s) fall outside.

## fig1b-above-chance

> Animals performed above 50% correct on delay only trials (65 +/- 2%) and on switch trials (74 +/- 2%).

In [ ]:
def windows(F, types, w):
    """Rolling proportion correct over w consecutive trials of one type within a session
    (pandas rolling, min_periods = w) [claims fig1b-window-performance; code behavior_analyzer]."""
    out = []
    for (an, se, ut), g in F[F.update_type.isin(types)].sort_values("start_time").groupby(["animal", "session", "update_type"]):
        v = g.correct.astype(float).rolling(w, min_periods=w).mean().dropna()
        out.append(pd.DataFrame(dict(animal=an, session=se, update_type=ut, prop=v.to_numpy())))
    return pd.concat(out, ignore_index=True)

sess_pc = F.groupby(["animal", "session", "update_type"]).correct.mean().reset_index()
for ut, e in ((1, 0.65), (2, 0.74)):
    x = sess_pc[sess_pc.update_type == ut]
    w = stats.wilcoxon(x.correct - 0.5)
    m_s, se_s = x.correct.mean(), x.correct.std(ddof=1) / np.sqrt(len(x))
    an = x.groupby("animal").correct.mean()
    print(f"{P.TRIAL_TYPES[ut]}: (a) sessions n={len(x)}, Wilcoxon W={w.statistic:.1f}, p={w.pvalue:.3g}; "
          f"(c) mean over sessions {m_s:.3f} +/- {se_s:.3f} SEM; (d) mean over animals {an.mean():.3f} +/- {an.std(ddof=1)/np.sqrt(len(an)):.3f} "
          f"(expected {e} +/- 0.039)")
    W40 = windows(F, [ut], 40)
    w2 = stats.wilcoxon(W40.prop - 0.5)
    print(f"   (b) 40-trial windows n={len(W40)}, W={w2.statistic:.1f}, p={w2.pvalue:.3g}, mean {W40.prop.mean():.3f}")
print("\nper animal proportion correct (mean of sessions):")
print(sess_pc.groupby(["animal", "update_type"]).correct.mean().unstack().round(3))

**Result: reproduced.** One proportion per session and trial type (reading a): delay only mean 0.649 +/- 0.017 over 66 sessions, Wilcoxon W = 125.5, p = 6.1e-10 (the paper prints W = 125.5, p = 6.197e-10); switch 0.745 +/- 0.020, W = 48.5, p = 3.3e-11 (paper W = 48.5). The means over animals (0.641 and 0.745) are also within tolerance. The 40-trial windows (reading b) also give p far below 0.05.

## fig1b-window-performance

> Over rolling windows of trials, proportion correct averaged 0.68 on delay only and 0.78 on switch trials.

In [ ]:
exp = {1: dict(mean=(0.68, 0.068), median=(0.67, 0.067)), 2: dict(mean=(0.78, 0.0196), median=(0.80, 0.08))}
for w in (40, 15):
    for ut in (1, 2):
        Wd = windows(F, [ut], w)
        mn, md_ = Wd.prop.mean(), Wd.prop.median()
        ok = abs(mn - exp[ut]["mean"][0]) <= exp[ut]["mean"][1] and abs(md_ - exp[ut]["median"][0]) <= exp[ut]["median"][1]
        print(f"window {w}: {P.TRIAL_TYPES[ut]} n windows={len(Wd)} mean {mn:.3f} median {md_:.3f} "
              f"(expected mean {exp[ut]['mean'][0]}, median {exp[ut]['median'][0]}); within tolerance: {ok}; "
              f"by animal mean {Wd.groupby('animal').prop.mean().round(2).to_dict()}")

**Result: reproduced.** With 15-trial windows (reading b) the window counts equal the paper's (6252 delay only, 1002 switch), and the means and medians are 0.678 and 0.667 (delay only) and 0.775 and 0.800 (switch), all within tolerance. With 40-trial windows (reading a) there are 4636 and 239 windows and the switch mean, 0.810, is outside tolerance.

## fig5a-stay-performance

> Animals performed very well on stay trials (proportion correct 0.92 over windows).

In [ ]:
for w in (40, 15):
    Wd = windows(F, [3], w)
    if len(Wd):
        print(f"window {w}: stay n windows={len(Wd)} mean {Wd.prop.mean():.3f} median {Wd.prop.median():.3f} "
              f"(expected mean 0.92 +/- 0.0196, median 0.93); within: {abs(Wd.prop.mean() - 0.92) <= 0.0196}")
    else:
        print(f"window {w}: no session has {w} stay trials")
x = sess_pc[sess_pc.update_type == 3]
print("per-session stay proportion correct: mean %.3f, median %.3f, n sessions %d; pooled over trials %.3f"
      % (x.correct.mean(), x.correct.median(), len(x), F[F.update_type == 3].correct.mean()))
print("stay trials per session: max", F[F.update_type == 3].groupby("session").size().max())

**Result: reproduced.** With 15-trial windows there are 117 stay windows, the paper's count, with mean 0.921 and median 0.933. No session has 40 stay trials, so the 40-trial reading gives no window.

## fig1d-heading-switch

> On switch trials the animals' heading (view angle) shifts from the initial toward the new side after the update cue, while on delay only trials it stays toward the initial side.

View angle is in radians (see the data check). Positive view angle goes with movement toward negative x (the left arm, `turn_type` 1), as the sign of the correlation above shows, and as the paper's code assumes when it negates view angle on right trials. "Toward the initial side" is therefore a positive angle on `turn_type` 1 trials and a negative one on `turn_type` 2 trials.

In [ ]:
edges = R[0]["va_edges"]; centers = (edges[:-1] + edges[1:]) / 2
traj = []
for r in R:
    tr = r["trials"]
    for j, idx in enumerate(r["va_traj_trials"]):
        traj.append((r["subject"], tr.loc[idx, "update_type"], tr.loc[idx, "turn_type"], r["va_traj"][j]))
TR = pd.DataFrame(traj, columns=["animal", "update_type", "turn_type", "traj"])
b_update = int(np.digitize(145.125, edges) - 1)          # bin of the update cue onset position
b_last = len(centers) - 1                                 # last central-arm bin before the choice point (y < 255)
res = {}
for tt, sgn in ((1, 1), (2, -1)):
    for ut in (1, 2):
        A_ = np.stack(TR[(TR.turn_type == tt) & (TR.update_type == ut)].traj.to_numpy())
        m = np.nanmean(A_, 0) * sgn       # positive = toward the initial side
        res[(tt, ut)] = m
        print(f"turn_type {tt} {P.TRIAL_TYPES[ut]:10s} n={len(A_)}: toward-initial view angle at update cue bin (y {centers[b_update]:.0f}) "
              f"{m[b_update]:+.3f} rad, in last central bin (y {centers[b_last]:.0f}) {m[b_last]:+.3f} rad; range of the trajectory {m.min():+.3f} to {m.max():+.3f}")
ok = all(res[(tt, 2)][b_update] > 0 and res[(tt, 2)][b_last] < 0 and res[(tt, 1)][b_last] > 0 for tt in (1, 2))
print("criterion holds for both sides:", ok)
print("\nper animal, switch trials, toward-initial angle in the last central bin:")
for an, g in TR[TR.update_type == 2].groupby("animal"):
    v = [np.nanmean(np.stack(g[g.turn_type == tt].traj.to_numpy()), 0)[b_last] * s for tt, s in ((1, 1), (2, -1)) if (g.turn_type == tt).any()]
    print(an, np.round(v, 3))
fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for k, tt in enumerate((1, 2)):
    for ut, col in ((1, "k"), (2, "m")):
        ax[k].plot(centers, res[(tt, ut)], col, label=P.TRIAL_TYPES[ut])
    ax[k].axvline(145.125, ls=":"); ax[k].axhline(0, c="grey", lw=0.5); ax[k].set_title(f"turn_type {tt}"); ax[k].set_xlabel("y position")
ax[0].set_ylabel("view angle toward initial side (rad)"); ax[0].legend(); plt.show()

**Result: reproduced.** On switch trials the mean view angle points toward the initial side at the update cue position (+0.198 and +0.183 rad for the two sides) and toward the new side in the last central-arm bin (-0.318 and -0.154 rad), while on delay only trials it points toward the initial side there (+0.197 and +0.217 rad). Per animal, 12 of the 14 animal-side means in the last bin point to the new side on switch trials.

## Position decoding: shared definitions

Encoder from 80% of the final-task delay only trials (S3), decoding of the held-out delay only trials and of all switch and stay trials in 200 ms bins aligned to the update cue (pseudo update on delay only trials; S4, S5). The bins are those of the code's aligned grid (`_align_by_times`, samples at -2.5, -2.3, ..., -0.1, 0.1, ..., 2.5 s): 26 bins of 0.2 s from -2.6 to 2.6 s whose edges fall on the cue, so no bin straddles it. Windows take bins by centre as the code does (`calc_trial_by_trial_quant_data`): (lo, hi] after the cue, [lo, hi) before it, so 0 to 1.5 s is the 8 bins centred 0.1 to 1.5 s and -1.5 to 0 s the 8 bins centred -1.5 to -0.1 s (printed below). In round 1 one bin was centred on the cue and counted after it; that grid is kept as a variant (section "Variant: the round-1 time grid"). Sessions are included per region with at least 20 units and at least 50 final-task trials of the session, warm-up trials left out (S1, as a reader amended it on 2026-11-02 in answer to this attempt's question; Methods, Decoding analyses: "less than 50 trials for the behavioral session"). This rule decides (`rule="final"`). The code's count of the encoder training trials (`_meets_exclusion_criteria`), which S1 keeps as a named variant, is shown beside every test (`rule="train"`). The trial counts under each rule are printed against the paper's.

Statistics (S9): a linear mixed model with random intercepts for animal and session within animal, with one predictor combining the grouping factors. The ANOVA is a Wald test of that predictor; statsmodels has no Kenward-Roger degrees of freedom. Pairwise contrasts of the predictor's levels are Tukey-adjusted with the studentized range over all levels, as emmeans `pairs()` does.

In [ ]:
W = dict(init=("init", 0, 1.5), new=("new", 0, 1.5), init_pre=("init", -1.5, 0), new_pre=("new", -1.5, 0),
         retro=("retro", 0, 1.5), init05=("init", 0, 0.5), new05=("new", 0, 0.5), init1=("init", 0, 1), new1=("new", 0, 1),
         init051=("init", 0.5, 1), new051=("new", 0.5, 1),
         init_front=("init_front", 0, 1.5), init_back=("init_back", 0, 1.5), new_front=("new_front", 0, 1.5), new_back=("new_back", 0, 1.5),
         init_front_occ=("init_front_occ", 0, 1.5), init_back_occ=("init_back_occ", 0, 1.5), new_front_occ=("new_front_occ", 0, 1.5), new_back_occ=("new_back_occ", 0, 1.5))
TAB = {(reg, rule): P.trial_table(R, reg, W, rule) for reg in ("CA1", "PFC") for rule in ("final", "train")}
for (reg, rule), T in TAB.items():
    c = T[T.correct == 1]
    print(f"{reg} rule={rule}: {T.session.nunique()} sessions, {T.animal.nunique()} animals; correct trials "
          f"{c.type.value_counts().to_dict()}; incorrect {T[T.correct == 0].type.value_counts().to_dict()}")
# the bins each window averages (bin centres, s), and the feature bins of each zone
tms = R[0]["dec"]["CA1"]["times"]
print("bin centres of the update-aligned grid:", tms.tolist())
for name, (zone, lo, hi) in W.items():
    if zone in ("init", "retro") or name == "init_front":
        print(f"  window {lo} to {hi} s: bin centres {tms[P.window_mask(tms, lo, hi)].tolist()}")
zb = pd.DataFrame([{k: str(np.where(r["geo"][k])[0].tolist()) for k in ("left", "right", "left_front", "left_back", "right_front", "right_back",
                     "left_front_occ", "left_back_occ", "right_front_occ", "right_back_occ", "central", "home_window", "retro", "gap")}
                   for r in R if "left_front" in r["geo"]])
for k in zb.columns:
    print(f"  zone {k}: feature bins {zb[k].value_counts().to_dict()} (sessions per configuration)")
print("paper: CA1 1295 switch, 926 delay only, 551 stay correct, 407 incorrect switch, 56 sessions; "
      "PFC 1109 switch, 798 delay only, 483 stay correct, 359 incorrect switch, 47 sessions")

def arm_model(T, types, cols=("init", "new"), outcome=(1,), factor="type", name="arm"):
    d = T[T.update_type.isin(types) & T.correct.isin(outcome)]
    L = P.long_format(d, list(cols), name=name)
    L["pred"] = L[factor].astype(str) + "|" + L[name]
    return P.lme(L, "value", "pred"), L

def report(res, pairs, label=""):
    print(f"  {label} ANOVA F({res['df1']}) = {res['F']:.2f}, p = {res['p_anova']:.3g}; n per level {res['n'].to_dict()}")
    print("  model means:", res["emm"].round(4).to_dict())
    out = {}
    for a, b in pairs:
        c = P.contrast(res, a, b)
        out[(a, b)] = c
        ps = f"{c['p_tukey']:.3g}" if c["p_tukey"] > 0 else "< 1e-15 (below the precision of the studentized range)"
        print(f"    {a} - {b}: estimate {c['estimate']:+.4f}, z = {c['z']:+.2f}, Tukey p = {ps} "
              f"(on {c['df_lo']} df: {c['p_tukey_lowdf']:.3g})")
    return out

## tuning-tiles-track

> Spatial tuning curves of hippocampal and prefrontal neurons tile the virtual track.

In [ ]:
for reg, rule in (("CA1", "final"), ("PFC", "final"), ("CA1", "train"), ("PFC", "train")):
    peaks = []
    for r in R:
        if not P.included(r, reg, rule):
            continue
        d = r["dec"][reg]
        peaks += list(d["tc_peak"][d["tc_active"]])
        gaps = np.where(r["geo"]["gap"])[0].tolist()
    cnt = np.bincount(peaks, minlength=P.N_BINS)
    track = [b for b in range(P.N_BINS) if b not in gaps]
    empty_track = [b for b in track if cnt[b] == 0]
    # longest run of empty bins among the track bins (gap bins between segments excluded)
    run, best = 0, 0
    for b in range(P.N_BINS):
        if b in gaps:
            run = 0; continue
        run = run + 1 if cnt[b] == 0 else 0; best = max(best, run)
    print(f"{reg} rule={rule}: {len(peaks)} units with spikes in the encoder; bins wholly inside the gaps between track segments: {gaps}")
    print(f"   units peaking per bin: {cnt.tolist()}")
    print(f"   (a) bins that are no unit's peak: all 50 bins -> {[b for b in range(P.N_BINS) if cnt[b] == 0]}; track bins only -> {empty_track}")
    print(f"   (b) longest run of empty track bins: {best}; peaks in central arm {int(cnt[:34].sum())}, goal arms {int(cnt[34:].sum())}")

**Result: reproduced.** In both regions every track bin is the peak of at least one unit (CA1 3404 units, PFC 2148 units), and the only bins that are no unit's peak are the four bins lying wholly in the gaps between track segments (34, 35, 42, 43), which hold no position. Peaks fall about equally on the central arm and the goal arms (CA1 1730 and 1674, PFC 1075 and 1073). The code's count of trials gives the same (CA1 2854, PFC 1790 units, no empty track bin).

## decoding-validation

> Hippocampal and prefrontal population activity decodes the animal's current position across the track.

In [ ]:
rng = np.random.default_rng(101)
for reg, rule in (("CA1", "final"), ("PFC", "final"), ("CA1", "train"), ("PFC", "train")):
    rows, confs = [], []
    for r in R:
        if not P.included(r, reg, rule):
            continue
        d = r["dec"][reg]
        tb = d["val_occ_bins"]; track = np.unique(tb)
        null = np.abs(rng.choice(track, size=len(tb)) - tb)
        rows.append(dict(session=r["session"], animal=r["subject"], err_bins=np.median(d["val_err_bins"]), null_bins=np.median(null),
                         err_units=np.median(d["val_err"]), err_moving=np.median(d["val_err_bins"][d["val_moving"]]),
                         moving_share=d["val_moving"].mean()))
        cf_ = d["val_conf"]; col = cf_.sum(0)
        with np.errstate(invalid="ignore"):
            confs.append(cf_ / col)
    V = pd.DataFrame(rows)
    a_ok = (V.err_bins < V.null_bins).mean()
    C = np.nanmean(np.stack(confs), 0)
    gaps = np.where(R[0]["geo"]["gap"])[0]
    occ = np.array([b for b in np.where(np.nansum(C, 0) > 0)[0] if b not in gaps])
    near = [abs(int(np.nanargmax(C[:, b])) - b) <= 2 for b in occ]
    print(f"{reg} rule={rule}: {len(V)} sessions; median error {V.err_bins.median():.1f} bins ({V.err_units.median():.1f} linearized units), "
          f"uniform null {V.null_bins.median():.1f} bins; (a) sessions with error below null: {int((V.err_bins < V.null_bins).sum())} of {len(V)} ({a_ok:.2f})")
    print(f"   (b) occupied track bins (gap bins excluded) whose session-averaged confusion column peaks within 2 bins: {sum(near)} of {len(occ)} ({np.mean(near):.2f})")
    print(f"   variant, moving bins only (as the encoder): median error {V.err_moving.median():.1f} bins; sessions below the null: "
          f"{int((V.err_moving < V.null_bins).sum())} of {len(V)}; share of bins moving: median {V.moving_share.median():.2f}")
    print("   per animal median error (bins):", V.groupby("animal").err_bins.median().to_dict())

**Result: reproduced.** On held-out delay only trials, the median error is below the uniform null in 52 of 55 CA1 sessions and 46 of 46 PFC sessions (median 9 and 7 bins against 16), and the session-averaged confusion matrix peaks within 2 bins of the true bin in 37 of 46 track bins for CA1 and 35 of 46 for PFC. Both readings hold for both regions, and under the code's count of trials too (44 of 47 and 37 of 37 sessions; 36 and 41 of 46 bins). Scoring only the bins where the animal moves, as the encoder uses, changes nothing (median 9 and 7 bins; 52 of 55 and 46 of 46 sessions); 97% of bins are moving bins (median over sessions). The error is lowest with the position at the same time, not 2 s earlier or later, in 52 of 55 CA1 and 43 of 46 PFC sessions (data check).

## fig2e-hpc-goal-increase

> After the update cue, hippocampal decoding of both the initial and the new goal arm is higher on switch than on delay only trials.

In [ ]:
pairs = [("switch|init", "delay only|init"), ("switch|new", "delay only|new")]
for rule in ("final", "train"):
    T = TAB[("CA1", rule)]
    for lab, types in (("(a) switch and delay only", [1, 2]), ("(b) with stay", [1, 2, 3])):
        print(f"rule={rule} {lab}")
        res, L = arm_model(T, types)
        report(res, pairs)
c = TAB[("CA1", "final")]; c = c[c.correct == 1]
print("\ntrial means (rule=final):", c.groupby("type")[["init", "new"]].mean().round(4).to_dict())
print("per animal, switch minus delay only (init, new):")
g = c.groupby(["animal", "type"])[["init", "new"]].mean().unstack()
print((g.xs("switch", axis=1, level=1) - g.xs("delay only", axis=1, level=1)).round(3))

**Result: reproduced.** Under both readings of the model and both inclusion rules, the ANOVA and both switch against delay only contrasts give p < 2e-10 with switch higher (Methods' count of trials, reading a: ANOVA p = 9.9e-22; initial +0.039, z = 7.47, Tukey p = 2.4e-12; new +0.035, z = 6.74, p = 1.1e-10). The trial means (switch initial 0.132 and new 0.135, delay only 0.093 and 0.100) are all within tolerance of 0.13, 0.14, 0.10 and 0.11. There are 1241 switch and 871 delay only correct trials in 55 sessions, against the paper's 1295 and 926 in 56. The increase holds in 4 of 7 animals for the initial arm and 5 of 7 for the new arm. On the round-1 grid (variant below) the contrasts give p = 2.7e-12 and 1.8e-7.

## fig2e-hpc-arms-equal

> Within trials, hippocampal decoding of the initial and the new arm after the update cue does not differ: the initial minus new difference is not different from zero and does not differ between switch and delay only trials.

In [ ]:
def diff_tests(T, types=(1, 2), cols=("init", "new"), outcome=(1,)):
    d = T[T.update_type.isin(types) & T.correct.isin(outcome)].copy()
    d["diff"] = d[cols[0]] - d[cols[1]]
    res = P.lme(d, "diff", "type")
    print(f"  LME diff ~ trial type: F({res['df1']}) = {res['F']:.2f}, p = {res['p_anova']:.3g}; means {res['emm'].round(4).to_dict()}")
    out = dict(p_lme=res["p_anova"])
    for ut in types:
        s = d[d.update_type == ut].groupby("session")["diff"].mean()
        w = stats.wilcoxon(s)
        print(f"  {P.TRIAL_TYPES[ut]}: trial mean diff {d[d.update_type == ut]['diff'].mean():+.4f} (n={int((d.update_type == ut).sum())}); "
              f"per-session Wilcoxon W = {w.statistic:.0f}, p = {w.pvalue:.3g}, n = {len(s)} sessions")
        out[ut] = (d[d.update_type == ut]["diff"].mean(), w.pvalue)
    return out
for rule in ("final", "train"):
    print("rule =", rule); diff_tests(TAB[("CA1", rule)])

**Result: reproduced.** The LME on the initial minus new difference gives p = 0.59 (p = 0.42 with the code's count of trials), and the per-session Wilcoxon tests over 55 sessions give p = 0.35 (delay only, mean -0.0071) and p = 0.39 (switch, mean -0.0032); none is below 0.05. Both means are within tolerance of -0.01. The round-1 grid gives p = 0.17, 0.27 and 0.44.

## sfig2e-goal-bias

> Within single 200 ms bins hippocampal goal decoding is strongly biased to one arm (43% of bins highly biased), yet within a trial both arms are represented (70% of trials have a mean goal bias index between -0.5 and 0.5).

In [ ]:
for rule, lo, hi, lab in (("final", 0, 1.5, "(a) 0 to 1.5 s"), ("final", 0, 2.5, "(b) 0 to 2.5 s"), ("train", 0, 1.5, "(a) 0 to 1.5 s"), ("train", 0, 2.5, "(b) 0 to 2.5 s")):
    bins_all, trial_means, n_tr = [], [], 0
    for r in R:
        if not P.included(r, "CA1", rule):
            continue
        d = r["dec"]["CA1"]; m = (d["update_type"] == 2) & (d["correct"] == 1)
        tm = P.window_mask(d["times"], lo, hi)
        I, N = d["zones"]["init"][m][:, tm], d["zones"]["new"][m][:, tm]
        with np.errstate(invalid="ignore", divide="ignore"):
            g = (N - I) / (N + I)
        bins_all.append(g.ravel()); trial_means.append(np.nanmean(g, 1)); n_tr += m.sum()
    b = np.concatenate(bins_all); b = b[np.isfinite(b)]; tmn = np.concatenate(trial_means); tmn = tmn[np.isfinite(tmn)]
    f1 = np.mean(np.abs(b) > 0.93); f2 = np.mean(np.abs(tmn) <= 0.5)
    print(f"rule={rule} {lab}: {n_tr} correct switch trials, {int(P.window_mask(d['times'], lo, hi).sum())} bins per trial, {len(b)} bins; fraction of bins |index| > 0.93: {f1:.3f} (expected 0.43 +/- 0.043); "
          f"fraction of trials with mean index in [-0.5, 0.5]: {f2:.3f} (expected 0.70 +/- 0.07); both within: {abs(f1-0.43)<=0.043 and abs(f2-0.7)<=0.07}")

**Result: reproduced.** Over 0 to 1.5 s (reading a, 8 bins per trial), 46.5% of bins of 1241 correct switch trials have an index beyond +/-0.93 and 64.8% of trials have a mean index between -0.5 and 0.5; over 0 to 2.5 s (13 bins), 46.1% and 70.4%. Both readings are within tolerance, as they are under the code's count of trials (45.9% and 64.9%; 45.9% and 70.8%).

## sfig3c-goals-over-retrospective

> On switch trials after the update cue, both goal arms are decoded more than retrospective non-local positions at the start of the track.

In [ ]:
for rule in ("final", "train"):
    T = TAB[("CA1", rule)]
    res, L = arm_model(T, [2], cols=("retro", "init", "new"))
    print("rule =", rule)
    report(res, [("switch|init", "switch|retro"), ("switch|new", "switch|retro")])
c = TAB[("CA1", "final")]; c = c[(c.correct == 1) & (c.update_type == 2)]
print("trial means (rule=final):", c[["retro", "init", "new"]].mean().round(4).to_dict(), "expected 0.12, 0.13, 0.14")

**Result: not reproduced.** On correct switch trials the goal arms exceed the retrospective zone only slightly and not significantly: initial minus retrospective +0.003 (Tukey p = 0.84), new minus retrospective +0.006 (p = 0.43); the ANOVA gives p = 0.46. With the code's count of trials p = 0.46 and 0.20, and on the round-1 grid p = 0.37 and 0.36. The trial means (retrospective 0.129, initial 0.132, new 0.135) are each within tolerance of the paper's 0.12, 0.13 and 0.14.

## sfig3c-retrospective-unchanged

> Retrospective non-local decoding after the update cue does not differ between switch and delay only trials.

In [ ]:
for rule in ("final", "train"):
    T = TAB[("CA1", rule)]
    res, L = arm_model(T, [1, 2], cols=("retro", "init", "new"))
    print("rule =", rule)
    report(res, [("switch|retro", "delay only|retro")])

**Result: not reproduced.** The retrospective zone is decoded more on switch than on delay only trials (+0.015, Tukey p = 0.036), below 0.05, so the paper's test finds a difference. The outcome depends on the time grid and on the inclusion rule: the code's count of trials gives +0.013, p = 0.14, and the round-1 grid, with one bin centred on the cue, gave +0.013, p = 0.12 (variant below). The degrees of freedom matter here too: on 49 degrees of freedom (sessions less levels, a deliberately small value in place of Kenward-Roger) the same contrast gives p = 0.050.

## sfig3ab-original-vs-update-cue

> Goal arm decoding also rises during the original cue, but then favours the back half of the goal arms, while after the update cue it favours the front half.

The original cue is aligned at trial start, 0 to 1.5 s after it (code `start_time` with no time before it). Each goal arm's bins are split as the code splits them (`quantify_aligned_data`): of an arm's n bins, `front` ends round(n / 2) bins before the arm's end and `rearhalf` starts n // 2 bins after its start, the front toward the choice point (lower linearized positions). The code's arm bins include a gap bin next to each arm, where the posterior is NaN; the same split over the bins with a posterior is shown as a variant. The bins of each zone are printed in the shared definitions.

In [ ]:
for rule, suf in (("final", ""), ("train", ""), ("final", "_occ")):
    Ts = P.trial_table(R, "CA1", W, rule, source="start"); Tu = TAB[("CA1", rule)]
    cols = [z + suf for z in ("init_front", "init_back", "new_front", "new_back")]
    Ts = Ts[(Ts.update_type == 2) & (Ts.correct == 1)].assign(cue="original"); Tu = Tu[(Tu.update_type == 2) & (Tu.correct == 1)].assign(cue="update")
    Lc = P.long_format(pd.concat([Ts, Tu], ignore_index=True)[["animal", "session", "trial", "cue"] + cols], cols, name="zone")
    Lc["pred"] = Lc.cue + "|" + Lc.zone
    res = P.lme(Lc, "value", "pred")
    print(f"rule = {rule}, halves {'of the occupied arm bins (variant)' if suf else 'as the code'}: original cue trials {len(Ts)}, update cue trials {len(Tu)} (paper 1264 and 1295)")
    report(res, [(f"original|{z}", f"update|{z}") for z in cols[1::2] + cols[0::2]])
    if rule == "final":
        means = pd.concat([Ts[cols].mean().rename("original"), Tu[cols].mean().rename("update")], axis=1).round(4)
        print(means)

**Result: partially reproduced.** With the arm halves split as the code splits them (feature bins 35 to 37 and 43 to 45 front, 38 to 41 and 46 to 49 back), the back halves of both arms are decoded more during the original cue than after the update cue (initial +0.033, z = 8.67; new +0.038, z = 10.04; Tukey p below the method's precision). The front halves are higher after the update cue, significantly for the new arm (-0.012, p = 0.044) but not for the initial arm (-0.010, p = 0.12), so three of the four contrasts hold. The code's count of trials gives front half p = 0.24 and 0.091. The code's front halves hold a gap bin (35, 43) with no posterior; split over the arm bins that have one (36 to 38 and 44 to 46 front, 39 to 41 and 47 to 49 back), the variant gives the same three of four (front: initial p = 0.10, new p = 0.0019), and its eight values (front 0.052, 0.062, 0.055, 0.070; back 0.103, 0.070, 0.106, 0.065) are all within tolerance, where those of the code's split are not (front 0.034 to 0.050 against 0.05 to 0.07; back 0.120 and 0.123 during the original cue against 0.10 and 0.11, 0.088 and 0.085 after the update cue against 0.07). There are 1241 trials per cue (paper 1264 and 1295). The new arm front contrast also depends on the degrees of freedom: on 47 (sessions less levels, a deliberately small value) it gives p = 0.063, and the variant split's p = 0.0019 gives 0.0056.

## sfig2f-all-trials-encoder

> With all trial types in the encoding model, the increase of hippocampal goal arm decoding after the update cue on switch trials remains similar.

In [ ]:
for rule in ("final", "train"):
    for k, lab in (("a", "(a) encoder from all trials, all decoded"), ("b", "(b) encoder from 80% of all types, held-out 20% decoded")):
        rows = []
        for r in R:
            if not P.included(r, "CA1", rule) or "allenc" not in r:
                continue
            d = r["allenc"][k]
            rows.append(pd.DataFrame(dict(animal=r["subject"], session=r["session"], update_type=d["update_type"], correct=d["correct"],
                                          init=P.window_mean(d["init"], d["times"], 0, 1.5), new=P.window_mean(d["new"], d["times"], 0, 1.5))))
        T2 = pd.concat(rows, ignore_index=True); T2["type"] = T2.update_type.map(P.TRIAL_TYPES)
        for outc, olab in (((1,), "(c) correct"), ((0, 1), "(d) all outcomes")):
            n = T2[T2.correct.isin(outc)].type.value_counts().to_dict()
            print(f"rule={rule} {lab} {olab}: trials {n} (paper 1322 switch, 4538 delay only)")
            res, L = arm_model(T2, [1, 2], outcome=outc)
            report(res, [("switch|init", "delay only|init"), ("switch|new", "delay only|new")])

**Result: partially reproduced.** With the encoder built from all trials and every trial decoded (reading a), both arms are higher on switch than on delay only trials for correct trials (initial +0.013, Tukey p = 9.0e-4; new +0.026, p = 1.9e-12) and for all outcomes (p = 1.6e-5 and 1.0e-6). With an 80/20 split of all trial types (reading b), only the new arm differs (p = 3.8e-6; all outcomes p = 2.2e-5); the initial arm does not (p = 0.44; all outcomes p = 0.19). The criterion asks for every reading, so the claim holds in part. The code's count of trials gives the same pattern (reading b, initial arm p = 0.33 and 0.066). Reading (a) has 1241 switch and 4225 delay only correct trials, closer to the paper's 1322 and 4538 than reading (b) (269 and 848).

## sfig5f-pfc-wider-spread

> Decoded position distributions are spatially wider in prefrontal cortex than in CA1.

Sessions with at least 20 units in each region; the region with more units is subsampled at random to the other's count (seeded per session), and both are decoded with the same encoder trials. Spread per bin follows the paper's code (`plot_region_error_distribution`, checked at the pinned commit: the outermost bins at or above the peak minus one standard deviation, before and after the peak); bins where the animal is at or past 120.35, delay only test trials of both outcomes. The verifier read the Methods' "how far in front and behind of the peak value the decoding results dropped below this threshold" as the first crossing on each side; that reading is shown as a variant.

In [ ]:
def equal_table(rule):
    rows = []
    for r in R:
        if "equal" not in r or (r["n_train"] if rule == "train" else len(r["trials"])) < P.MIN_TRIALS:
            continue
        for reg in ("CA1", "PFC"):
            d = r["equal"][reg]
            sp = np.where(d["pos"] >= 120.35, d["spread"], np.nan)
            with np.errstate(all="ignore"):
                rows.append(pd.DataFrame(dict(animal=r["subject"], session=r["session"], region=reg, update_type=d["update_type"],
                                              correct=d["correct"], spread=np.nanmean(sp, 1),
                                              spread_first=np.nanmean(np.where(d["pos"] >= 120.35, d["spread_first"], np.nan), 1),
                                              init_pre=P.window_mean(d["init"], d["times"], -2, 0),
                                              new_pre=P.window_mean(d["new"], d["times"], -2, 0), n_units=d["n_units"])))
    return pd.concat(rows, ignore_index=True)
binw = np.diff(R[0]["geo"]["edges"])[0]
for rule in ("final", "train"):
    E = equal_table(rule)
    d = E[E.update_type == 1].dropna(subset=["spread"])
    res = P.lme(d, "spread", "region")
    m = d.groupby("region").spread.mean()
    print(f"rule={rule}: {d.session.nunique()} sessions, units per region per session {E.groupby('session').n_units.first().min()} to {E.groupby('session').n_units.first().max()}, "
          f"trials per region {d.groupby('region').size().to_dict()} (paper 829 trials)")
    print(f"   (a) feature bins: CA1 {m['CA1']:.3f}, PFC {m['PFC']:.3f} (expected 2.4 +/- 0.25, 3.13 +/- 0.27); LME region p = {res['p_anova']:.3g}, PFC - CA1 = {res['emm']['PFC'] - res['emm']['CA1']:+.3f}")
    print(f"   (b) linearized units (bin width {binw:.2f}): CA1 {m['CA1'] * binw:.2f}, PFC {m['PFC'] * binw:.2f}")
    rf = P.lme(d.dropna(subset=["spread_first"]), "spread_first", "region"); mf = d.groupby("region").spread_first.mean()
    print(f"   variant, first crossing on each side of the peak (Methods wording): CA1 {mf['CA1']:.3f}, PFC {mf['PFC']:.3f} bins; "
          f"LME region p = {rf['p_anova']:.3g}, PFC - CA1 = {rf['emm']['PFC'] - rf['emm']['CA1']:+.3f}")
    if rule == "final":
        print("   per animal (CA1, PFC):", d.groupby(["animal", "region"]).spread.mean().unstack().round(2).to_dict("index"))

**Result: partially reproduced.** With equal cell numbers in 42 sessions (1026 delay only trials per region), PFC spread exceeds CA1 under the code's rule (4.37 against 4.07 feature bins, LME p = 0.0031), but both values are outside the tolerances of the paper's 3.13 and 2.4 bins. PFC exceeds CA1 in 5 of 7 animals. The code's count of trials (36 sessions) gives 4.44 against 4.05, p = 1.3e-4. The first-crossing reading of the Methods' wording also gives PFC wider (0.37 against 0.14 bins, p = 1.8e-122), with values much further from the paper's: at that threshold the run of bins around the peak is mostly the peak alone. Both readings give the same outcome.

## sfig5g-ca1-more-goal-coding

> Before the update cue, goal arms are decoded more strongly from CA1 than from prefrontal cortex.

In [ ]:
for rule in ("final", "train"):
    E = equal_table(rule)
    d = E[(E.update_type == 1) & (E.correct == 1)]
    L = P.long_format(d[["animal", "session", "region", "init_pre", "new_pre"]], ["init_pre", "new_pre"], name="arm")
    L["pred"] = L.region + "|" + L.arm
    res = P.lme(L, "value", "pred")
    print(f"rule={rule}: correct delay only trials per region {d.groupby('region').size().to_dict()} (paper 364)")
    report(res, [("CA1|init_pre", "PFC|init_pre"), ("CA1|new_pre", "PFC|new_pre")])
    print("   trial means:", d.groupby("region")[["init_pre", "new_pre"]].mean().round(4).to_dict("index"),
          "(expected PFC init 0.06, CA1 new 0.11, PFC new 0.05)")

**Result: reproduced.** Before the pseudo update time (-2 to 0 s), CA1 decodes both arms more than PFC with equal cell numbers (initial +0.057, z = 13.4; new +0.061, z = 14.4; Tukey p at the method's precision floor; 717 trials per region, paper 364). The PFC initial (0.051), CA1 new (0.110) and PFC new (0.049) means are within tolerance. The code's count of trials gives the same (z = 12.8 and 14.1).

## fig2g-pfc-no-goal-increase

> Prefrontal position decoding of the goal arms after the update cue does not differ between switch and delay only trials, and the two arms are represented equally.

In [ ]:
pairs = [("switch|init", "delay only|init"), ("switch|new", "delay only|new")]
for rule in ("final", "train"):
    T = TAB[("PFC", rule)]
    for lab, types in (("(a) switch and delay only", [1, 2]), ("(b) with stay", [1, 2, 3])):
        print(f"rule={rule} {lab}")
        res, L = arm_model(T, types)
        report(res, pairs)
    print(f"rule={rule} within-trial difference")
    diff_tests(T)
c = TAB[("PFC", "final")]; c = c[c.correct == 1]
print("\ntrial means (rule=final):", c.groupby("type")[["init", "new"]].mean().round(4).to_dict(), "expected switch 0.07/0.07, delay only 0.06/0.06")

**Result: not reproduced.** In reading (a), switch and delay only trials in the model, the new arm is higher on switch than on delay only trials (+0.013, Tukey p = 0.016) and the ANOVA gives p = 0.017. In reading (b), with stay trials, the new arm contrast (p = 0.030) and the ANOVA (p = 0.025) are also below 0.05, so the outcome no longer depends on the reading. The initial arm contrasts (p = 0.61 and 0.82), the LME on the within-trial difference (p = 0.12) and the per-session Wilcoxon tests over 46 sessions (switch p = 0.055, delay only p = 0.33) are not significant. The effect is small, and the trial means (switch 0.069 and 0.073, delay only 0.066 and 0.062) are all within tolerance of the paper's. With the code's count of trials (37 sessions), reading (a) gives new arm p = 0.021 and ANOVA p = 0.018, and the switch Wilcoxon p = 0.044. On the round-1 grid reading (a) gave p = 0.035 and 0.038 and reading (b) none below 0.05 (new arm p = 0.066).

## sfig5e-pfc-windows

> Prefrontal goal arm decoding on switch and delay only trials does not differ in shorter windows after the update cue either.

In [ ]:
for rule in ("final", "train"):
    T = TAB[("PFC", rule)]
    for a, b, lab in (("init05", "new05", "0 to 0.5 s"), ("init1", "new1", "0 to 1 s"), ("init", "new", "0 to 1.5 s"), ("init051", "new051", "0.5 to 1 s")):
        res, L = arm_model(T, [1, 2], cols=(a, b))
        print(f"rule={rule} window {lab}")
        report(res, [(f"switch|{a}", f"delay only|{a}"), (f"switch|{b}", f"delay only|{b}")])

**Result: not reproduced.** The new arm is higher on switch than on delay only trials over 0 to 1 s (+0.013, Tukey p = 0.033), 0.5 to 1 s (+0.018, p = 0.029) and 0 to 1.5 s (+0.013, p = 0.016); only 0 to 0.5 s gives no difference (new arm p = 0.23, initial arm p = 1.0). The code's count of trials gives the same pattern (p = 0.040, 0.017 and 0.021; 0 to 0.5 s p = 0.35), and so does the round-1 grid (p = 0.019, 0.024 and 0.035). No initial arm contrast is significant.

## fig5d-stay-hpc-like-delay

> Hippocampal goal arm decoding after the second cue on stay trials does not differ from delay only trials.

In [ ]:
STAY = {}
for rule in ("final", "train"):
    T = TAB[("CA1", rule)]
    res, L = arm_model(T, [1, 2, 3])
    print(f"rule={rule}")
    STAY[rule] = report(res, [("stay|init", "delay only|init"), ("stay|new", "delay only|new"), ("stay|init", "switch|init"), ("stay|new", "switch|new")])
c = TAB[("CA1", "final")]; c = c[(c.correct == 1) & (c.update_type == 3)]
print("stay trial means (rule=final): init %.4f, new %.4f (expected 0.10, 0.12)" % (c.init.mean(), c.new.mean()))

**Result: reproduced.** Neither arm on stay trials differs from delay only trials: initial +0.008 (Tukey p = 0.79), new +0.015 (p = 0.17). The stay means (0.101 and 0.115) are within tolerance of 0.10 and 0.12; there are 530 correct stay trials (paper 551). The code's count of trials gives p = 0.90 and 0.10, and the round-1 grid gave p = 0.85 and 0.063.

## fig5d-stay-hpc-below-switch

> Hippocampal goal arm decoding after the second cue is lower on stay trials than on switch trials.

The model is the one fitted for fig5d-stay-hpc-like-delay; its stay against switch contrasts are printed there and repeated here.

In [ ]:
for rule in ("final", "train"):
    for k in [("stay|init", "switch|init"), ("stay|new", "switch|new")]:
        c = STAY[rule][k]
        print(f"rule={rule} {k[0]} - {k[1]}: estimate {c['estimate']:+.4f}, z = {c['z']:+.2f}, Tukey p = {c['p_tukey']:.3g}")

**Result: reproduced.** Both arms are lower on stay than on switch trials: initial -0.031 (Tukey p = 3.5e-6), new -0.020 (p = 0.0094); with the code's count of trials p = 1.8e-7 and 0.019. On the round-1 grid the new arm contrast was not significant (-0.012, p = 0.38), so the new arm part depends on the grid.

## sfig10-stay-transient-new-arm

> In the first 0.5 s after the stay cue, hippocampal decoding of the new arm is transiently higher on stay than on delay only trials, with no difference for the initial arm.

In [ ]:
for rule in ("final", "train"):
    T = TAB[("CA1", rule)]
    res, L = arm_model(T, [1, 2, 3], cols=("init05", "new05"))
    print(f"rule={rule}")
    report(res, [("stay|new05", "delay only|new05"), ("stay|init05", "delay only|init05")])
c = TAB[("CA1", "final")]; c = c[c.correct == 1]
print("trial means 0 to 0.5 s (rule=final):", c.groupby("type")[["init05", "new05"]].mean().round(4).to_dict(),
      "(expected stay new 0.13, delay only new 0.11, stay init 0.09)")

**Result: reproduced.** Over 0 to 0.5 s (bins centred 0.1, 0.3 and 0.5 s) the new arm is higher on stay than on delay only trials (+0.030, Tukey p = 0.0052; p = 0.0030 with the code's count of trials) and the initial arm does not differ (p = 0.93). The stay new arm (0.124) and stay initial arm (0.098) means are within tolerance; the delay only new arm mean (0.094) is below the tolerance of 0.11 +/- 0.011. The round-1 grid gave +0.024, p = 0.037.

## Residual goal codes (GLM)

Methods, "Generalized linear model (GLM) analysis": one GLM per goal representation (initial, new) with rotational and translational velocity as predictors and the integrated goal probability as response, fitted on the 200 ms bins over 2.5 s before and after the update cue "across all trial types and animals"; the Pearson residuals are the residual goal codes. The family is Poisson, as in the paper's code (`get_residuals`, statsmodels GLM). Rotational velocity is mirrored with the decoding (S5), so that its sign is relative to the initial side. The code's variant fits the GLM per session on the switch trials of the comparison; it is shown and does not decide.

In [ ]:
import statsmodels.api as sm
def bin_table(reg, rule="final", RR=None):
    rows = []
    for r in (R if RR is None else RR):
        if not P.included(r, reg, rule):
            continue
        d = r["dec"][reg]; n, k = d["zones"]["init"].shape
        sgn = np.where(d["turn_type"] == 1, 1.0, -1.0)[:, None]
        rows.append(pd.DataFrame(dict(animal=r["subject"], session=r["session"], trial=np.repeat(d["trial"], k),
                                      update_type=np.repeat(d["update_type"], k), correct=np.repeat(d["correct"], k),
                                      t=np.tile(d["times"], n), init=d["zones"]["init"].ravel(), new=d["zones"]["new"].ravel(),
                                      rot=(d["rot"] * sgn).ravel(), trans=d["trans"].ravel())))
    return pd.concat(rows, ignore_index=True)

def residuals(B, by_session=False, types=(1, 2, 3)):
    B = B[B.update_type.isin(types)].dropna(subset=["init", "new", "rot", "trans"]).copy()
    for arm in ("init", "new"):
        B[arm + "_res"] = np.nan
        groups = B.groupby("session") if by_session else [("all", B)]
        for _, g in groups:
            X = sm.add_constant(g[["rot", "trans"]].to_numpy())
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                f = sm.GLM(g[arm].to_numpy(), X, family=sm.families.Poisson()).fit()
            B.loc[g.index, arm + "_res"] = f.resid_pearson
    return B

def resid_trials(B, lo, hi):
    w = B[P.window_mask(B.t.to_numpy(), lo, hi)]
    T = w.groupby(["animal", "session", "trial", "update_type", "correct"])[["init_res", "new_res"]].mean().reset_index()
    T["outcome"] = np.where(T.correct == 1, "correct", "incorrect")
    return T

def outcome_model(T, ut, label):
    d = T[T.update_type == ut]
    L = P.long_format(d, ["init_res", "new_res"], name="arm"); L["pred"] = L.outcome + "|" + L.arm
    res = P.lme(L, "value", "pred")
    print(f"  {label}: trials {d.outcome.value_counts().to_dict()}")
    out = report(res, [("correct|new_res", "incorrect|new_res"), ("correct|init_res", "incorrect|init_res")])
    d = d.assign(diff=d.init_res - d.new_res)
    rd = P.lme(d, "diff", "outcome")
    print(f"    initial minus new by outcome: F = {rd['F']:.2f}, p = {rd['p_anova']:.3g}; means {rd['emm'].round(4).to_dict()}")
    print("    trial means:", d.groupby("outcome")[["init_res", "new_res"]].mean().round(4).to_dict("index"))
    out["diff_p"] = rd["p_anova"]
    return out

BC = bin_table("CA1")
BCt = bin_table("CA1", "train")
GLM = {"methods": residuals(BC), "code": residuals(BC, by_session=True, types=(2,)), "methods, rule=train": residuals(BCt)}
GLMd = {"code": residuals(BC, by_session=True, types=(1,))}
print("CA1 bins:", len(BC), "; trials:", BC.groupby(["session", "trial"]).ngroups, "; sessions:", BC.session.nunique())
print("rule=train variant: CA1 bins:", len(BCt), "; trials:", BCt.groupby(["session", "trial"]).ngroups, "; sessions:", BCt.session.nunique())

## fig6b-hpc-new-lower-incorrect

> On incorrect switch trials, hippocampal new goal arm decoding after the update cue is less elevated than on correct trials, after accounting for speed and rotation.

In [ ]:
G6 = {}
for k, B in GLM.items():
    print(f"GLM {k} ({'decides' if k == 'methods' else 'variant'}), 0 to 1.5 s")
    G6[(k, "post")] = outcome_model(resid_trials(B, 0, 1.5), 2, "switch trials")

**Result: reproduced.** With the Methods' GLM, the new arm residual is higher on correct than incorrect switch trials (+0.084, Tukey p = 3.7e-4; ANOVA p = 8.9e-5; 1241 correct and 395 incorrect trials), and the trial means (correct 0.073, incorrect 0.012) are within tolerance of 0.06 and 0.0. The code's per-session GLM gives the same (p = 9.5e-4), and so does the code's count of trials (p = 9.1e-4).

## fig6b-hpc-initial-no-difference

> Hippocampal initial goal arm decoding after the update cue does not differ between correct and incorrect switch trials.

The test is the initial arm contrast of the model above.

In [ ]:
for k in GLM:
    c = G6[(k, "post")][("correct|init_res", "incorrect|init_res")]
    print(f"GLM {k}: correct - incorrect, initial arm residual: {c['estimate']:+.4f}, z = {c['z']:+.2f}, Tukey p = {c['p_tukey']:.3g}")

**Result: reproduced.** With the Methods' GLM the initial arm residual does not differ between correct and incorrect switch trials (+0.041, Tukey p = 0.21). The code's per-session GLM, a variant, gives p = 0.25, and the code's count of trials p = 0.16.

## fig6-hpc-pre-cue-initial-higher-incorrect

> Before the update cue, hippocampal initial goal arm decoding is higher on incorrect than on correct switch trials, beyond motor differences.

In [ ]:
for k, B in GLM.items():
    print(f"GLM {k}, -1.5 to 0 s")
    G6[(k, "pre")] = outcome_model(resid_trials(B, -1.5, 0), 2, "switch trials")

**Result: reproduced.** Over -1.5 to 0 s the initial arm residual is higher on incorrect than correct switch trials (+0.067, Tukey p = 4.3e-4), while the new arm does not differ (p = 0.88). The code's count of trials gives the same (p = 3.1e-4); the code's per-session GLM gives the same direction, not significant (+0.035, p = 0.12).

## sfig12-delay-hpc-no-outcome-difference

> On delay only trials, hippocampal goal arm decoding in the update period does not differ between correct and incorrect trials.

Held-out delay only trials of both outcomes, aligned at the pseudo update time, 0 to 1.5 s.

In [ ]:
for k, B in (("methods", GLM["methods"]), ("code", GLMd["code"]), ("methods, rule=train", GLM["methods, rule=train"])):
    print(f"GLM {k}")
    outcome_model(resid_trials(B, 0, 1.5), 1, "delay only trials")

**Result: not reproduced.** With the Methods' GLM the initial arm residual is higher on incorrect than correct delay only trials (+0.055, Tukey p = 0.043); the new arm does not differ (p = 0.70) and the initial minus new difference does not differ by outcome (p = 0.18). One of the three tests gives p below 0.05. The code's per-session GLM, a variant, gives no p below 0.05 (0.33, 0.93 and 0.41), and neither does the code's count of trials (0.096, 0.95 and 0.12); the round-1 grid gave p = 0.012 for the initial arm. On 51 degrees of freedom (sessions less levels, a deliberately small value in place of Kenward-Roger) the initial arm contrast gives p = 0.053.

## fig7c-hpc-commitment-new-arm

> The more committed the animal was to the initial arm at the update cue (view angle toward it), the larger the post-cue increase in hippocampal decoding of the new arm.

In [ ]:
def commit_table(rule, RR=None):
  rows = []
  for r in (R if RR is None else RR):
    if not P.included(r, "CA1", rule):
        continue
    d = r["dec"]["CA1"]; m = (d["update_type"] == 2) & (d["correct"] == 1)
    sgn = np.where(d["turn_type"] == 1, 1.0, -1.0)
    z = d["zones"]
    rows.append(pd.DataFrame(dict(animal=r["subject"], session=r["session"], va=(d["va"] * sgn)[m],
                                  d_new=(P.window_mean(z["new"], d["times"], 0, 1.5) - P.window_mean(z["new"], d["times"], -1.5, 0))[m],
                                  d_init=(P.window_mean(z["init"], d["times"], 0, 1.5) - P.window_mean(z["init"], d["times"], -1.5, 0))[m])))
  CM = pd.concat(rows, ignore_index=True).dropna()
  return CM
CMS = {rule: commit_table(rule) for rule in ("final", "train")}
CM = CMS["final"]
CM["quartile"] = pd.qcut(CM.va, 4, labels=["Q1 toward new", "Q2", "Q3", "Q4 toward initial"])
for rule, cm in CMS.items():
    for arm in ("d_new", "d_init"):
        rho, p = stats.spearmanr(cm.va, cm[arm])
        print(f"rule={rule} {arm}: Spearman rho = {rho:+.4f}, p = {p:.3g}, n = {len(cm)} trials (paper n = 1235)")
print("quartile means:\n", CM.groupby("quartile", observed=True)[["va", "d_new", "d_init"]].mean().round(4))
print("per animal rho (new arm):", {a: round(stats.spearmanr(g.va, g.d_new)[0], 3) for a, g in CM.groupby("animal")})
print("view angle at the cue, toward initial (rad): range %.3f to %.3f" % (CM.va.min(), CM.va.max()))

**Result: partially reproduced.** The change in new arm decoding is positively correlated with view angle toward the initial arm (Spearman rho = +0.077, p = 0.0067, n = 1241), but rho is outside the tolerance of 0.1237 +/- 0.0124. The quartile means rise from 0.024 (toward new) to 0.073 and 0.052 in the two quartiles toward the initial arm. Per animal, rho is positive in 5 of 7. The code's count of trials gives rho = +0.087, p = 0.0031, and the round-1 grid gave rho = +0.057, p = 0.043.

## fig7c-hpc-commitment-initial-arm

> The change in hippocampal initial arm decoding after the update cue is not correlated with view angle at the cue.

In [ ]:
rho, p = stats.spearmanr(CM.va, CM.d_init)
print(f"initial arm: Spearman rho = {rho:+.4f}, p = {p:.3g}, n = {len(CM)}")
rho, p = stats.spearmanr(CMS["train"].va, CMS["train"].d_init)
print(f"variant rule=train, initial arm: Spearman rho = {rho:+.4f}, p = {p:.3g}, n = {len(CMS['train'])}")

**Result: reproduced.** The change in initial arm decoding is not correlated with view angle (rho = +0.030, p = 0.29, n = 1241), as in the paper (rho = 0.044, p = 0.12). The code's count of trials gives p = 0.29. On the round-1 grid the correlation was +0.058, p = 0.040, so this outcome depends on the grid.

## Variant: the round-1 time grid

In round 1 the update-aligned decoding used 25 bins of 0.2 s from -2.5 to 2.5 s, so that one bin (-0.1 to 0.1 s) was centred on the cue and counted in the windows after it, and windows took the bins whose centres fall in [lo, hi) (0 to 1.5 s was then -0.1 to 1.5 s). The verifier pointed out that this bin straddles the cue. The grid above follows the code (`_align_by_times`, samples at -2.5, -2.3, ..., -0.1, 0.1, ..., 2.5 s, windows as `calc_trial_by_trial_quant_data`). The cell below repeats the deciding test of each claim on update-aligned windows with the round-1 grid, so that the dependence of each outcome on the grid can be seen. It does not decide any outcome.

In [ ]:
# the deciding tests of the update-aligned claims, on the round-1 grid (variant; the grid above decides)
P.WINDOW_RULE = "round1"
rows1 = []
def add1(claim, test, c=None, p=None, est=None):
    rows1.append(dict(claim=claim, test=test, estimate=round(float(est if c is None else c["estimate"]), 4),
                      p=float(p if c is None else c["p_tukey"])))
def outcome_quiet(T, ut):
    d = T[T.update_type == ut]
    L = P.long_format(d, ["init_res", "new_res"], name="arm"); L["pred"] = L.outcome + "|" + L.arm
    return P.lme(L, "value", "pred")
try:
    R1 = [dict(r, dec={k: dict(v, **r["dec_old"][k]) for k, v in r["dec"].items()}) for r in R]
    print("round-1 grid, bin centres:", R1[0]["dec"]["CA1"]["times"].tolist())
    T1 = P.trial_table(R1, "CA1", W, "final")
    res, _ = arm_model(T1, [1, 2])
    add1("fig2e-hpc-goal-increase", "ANOVA", p=res["p_anova"], est=np.nan)
    for a in ("init", "new"):
        add1("fig2e-hpc-goal-increase", f"switch - delay only, {a}", P.contrast(res, f"switch|{a}", f"delay only|{a}"))
    d = T1[T1.update_type.isin([1, 2]) & (T1.correct == 1)].assign(diff=lambda x: x.init - x.new)
    rd = P.lme(d, "diff", "type")
    add1("fig2e-hpc-arms-equal", "LME initial minus new ~ trial type", p=rd["p_anova"], est=rd["emm"]["switch"] - rd["emm"]["delay only"])
    for ty in ("delay only", "switch"):
        sm_ = d[d.type == ty].groupby("session")["diff"].mean()
        add1("fig2e-hpc-arms-equal", f"Wilcoxon of session means, {ty}", p=stats.wilcoxon(sm_).pvalue, est=sm_.mean())
    res, _ = arm_model(T1, [2], cols=("retro", "init", "new"))
    for a in ("init", "new"):
        add1("sfig3c-goals-over-retrospective", f"{a} - retro, switch", P.contrast(res, f"switch|{a}", "switch|retro"))
    res, _ = arm_model(T1, [1, 2], cols=("retro", "init", "new"))
    add1("sfig3c-retrospective-unchanged", "retro, switch - delay only", P.contrast(res, "switch|retro", "delay only|retro"))
    res, _ = arm_model(T1, [1, 2, 3])
    for a in ("init", "new"):
        add1("fig5d-stay-hpc-like-delay", f"stay - delay only, {a}", P.contrast(res, f"stay|{a}", f"delay only|{a}"))
    for a in ("init", "new"):
        add1("fig5d-stay-hpc-below-switch", f"stay - switch, {a}", P.contrast(res, f"stay|{a}", f"switch|{a}"))
    res, _ = arm_model(T1, [1, 2, 3], cols=("init05", "new05"))
    for a in ("new05", "init05"):
        add1("sfig10-stay-transient-new-arm", f"stay - delay only, {a}", P.contrast(res, f"stay|{a}", f"delay only|{a}"))
    T1p = P.trial_table(R1, "PFC", W, "final")
    for lab, types in (("(a) switch and delay only", [1, 2]), ("(b) with stay", [1, 2, 3])):
        res, _ = arm_model(T1p, types)
        add1("fig2g-pfc-no-goal-increase", f"{lab}: ANOVA", p=res["p_anova"], est=np.nan)
        for a in ("init", "new"):
            add1("fig2g-pfc-no-goal-increase", f"{lab}: switch - delay only, {a}", P.contrast(res, f"switch|{a}", f"delay only|{a}"))
    d = T1p[T1p.update_type.isin([1, 2]) & (T1p.correct == 1)].assign(diff=lambda x: x.init - x.new)
    for ty in ("delay only", "switch"):
        sm_ = d[d.type == ty].groupby("session")["diff"].mean()
        add1("fig2g-pfc-no-goal-increase", f"Wilcoxon of session means, {ty}", p=stats.wilcoxon(sm_).pvalue, est=sm_.mean())
    for a, b, lab in (("init05", "new05", "0 to 0.5 s"), ("init1", "new1", "0 to 1 s"), ("init", "new", "0 to 1.5 s"), ("init051", "new051", "0.5 to 1 s")):
        res, _ = arm_model(T1p, [1, 2], cols=(a, b))
        for z in (a, b):
            add1("sfig5e-pfc-windows", f"{lab}: switch - delay only, {z}", P.contrast(res, f"switch|{z}", f"delay only|{z}"))
    G1 = residuals(bin_table("CA1", RR=R1))
    post1, pre1 = resid_trials(G1, 0, 1.5), resid_trials(G1, -1.5, 0)
    rr = outcome_quiet(post1, 2)
    add1("fig6b-hpc-new-lower-incorrect", "correct - incorrect, new, switch", P.contrast(rr, "correct|new_res", "incorrect|new_res"))
    add1("fig6b-hpc-initial-no-difference", "correct - incorrect, initial, switch", P.contrast(rr, "correct|init_res", "incorrect|init_res"))
    rr = outcome_quiet(pre1, 2)
    add1("fig6-hpc-pre-cue-initial-higher-incorrect", "correct - incorrect, initial, -1.5 to 0 s", P.contrast(rr, "correct|init_res", "incorrect|init_res"))
    rr = outcome_quiet(post1, 1)
    for a in ("init_res", "new_res"):
        add1("sfig12-delay-hpc-no-outcome-difference", f"correct - incorrect, {a}, delay only", P.contrast(rr, f"correct|{a}", f"incorrect|{a}"))
    CM1 = commit_table("final", RR=R1)
    for arm, cl in (("d_new", "fig7c-hpc-commitment-new-arm"), ("d_init", "fig7c-hpc-commitment-initial-arm")):
        rho, pp = stats.spearmanr(CM1.va, CM1[arm])
        add1(cl, f"Spearman rho, {arm}, n = {len(CM1)}", p=pp, est=rho)
finally:
    P.WINDOW_RULE = "code"
V1 = pd.DataFrame(rows1)
print(V1.to_string(formatters={"p": "{:.3g}".format}))


## Theta phase: shared definitions

CA1 decoding in 25 ms bins over 1.5 s before and after the update cue (held-out delay only, switch and stay trials, the encoder of the 200 ms analyses). As a variant, the theta phase stored in the file for that channel, which the paper's code reads (`general/lfp.py`), is referenced the same way and every theta test is repeated with it. Theta comes from the LFP of the channel the files mark as `ripple_channel` (the CA1 pyramidal-layer channel chosen by sharp-wave ripple power, Methods). Phase is set from peaks and troughs of the 4 to 12 Hz equiripple-filtered LFP, and 0 is put at the 30-degree bin of maximal CA1 pyramidal-cell firing in each session (Methods, "Theta cycles and phase"). The half cycles are -pi to 0 and 0 to pi.

The computed and the stored phase agree closely in most sessions (printed below). Where they disagree (lowest mean cosine -0.49, session S20-210511), the pyramidal spikes are barely locked to theta (0.085 to 0.086 of spikes in the peak 30-degree bin, against 0.083 for uniform firing), so the reference phase of maximal firing is nearly arbitrary and the two estimates pick different references.

In [ ]:
def theta_table(key, rule="final"):
  rows = []
  for r in R:
    if not P.included(r, "CA1", rule) or "theta" not in r:
        continue
    th = dict(r["theta"]); th["phase"] = th[key]
    for half, (lo, hi) in (("h1", (-np.pi, 0)), ("h2", (0, np.pi)), ("all", (-np.pi, np.pi))):
        pm = (th["phase"] > lo) & (th["phase"] <= hi)
        for win, (a, b) in (("post", (0, 1.5)), ("pre", (-1.5, 0))):
            tm = (th["times"] >= a) & (th["times"] < b)
            m = pm & tm[None, :]
            with np.errstate(all="ignore"):
                vals = {z: np.nanmean(np.where(m, th[z], np.nan), 1) for z in ("init", "new", "central", "home_window")}
            rows.append(pd.DataFrame(dict(animal=r["subject"], session=r["session"], trial=th["trial"], update_type=th["update_type"],
                                          correct=th["correct"], half=half, win=win, **vals)))
  out = pd.concat(rows, ignore_index=True); out["type"] = out.update_type.map(P.TRIAL_TYPES)
  return out
THS = {"computed": theta_table("phase"), "stored": theta_table("phase_stored"), "computed, rule=train": theta_table("phase", "train")}
TH = THS["computed"]
agree = [np.nanmean(np.cos(r["theta"]["phase"] - r["theta"]["phase_stored"])) for r in R if "theta" in r]
print("stored phase: unit as labelled", {r["theta"]["stored_unit"] for r in R if "theta" in r},
      "; value range %.3f to %.3f" % (min(r["theta"]["stored_range"][0] for r in R if "theta" in r), max(r["theta"]["stored_range"][1] for r in R if "theta" in r)))
print("agreement of computed and stored phase at the decoding bins, mean cos(difference) per session: median %.3f, min %.3f" % (np.median(agree), np.min(agree)))
AG = pd.DataFrame([dict(session=r["session"], agree=a, ref_computed=np.degrees(r["theta"]["ref"]),
                        stored_peak_share=r["theta"]["spike_phase_hist_stored"].max() / r["theta"]["spike_phase_hist_stored"].sum(),
                        computed_peak_share=r["theta"]["spike_phase_hist"].max() / r["theta"]["spike_phase_hist"].sum())
                   for r, a in zip([r for r in R if "theta" in r], agree)]).sort_values("agree")
print("sessions with the lowest agreement (spike phase locking: share of pyramidal spikes in the peak 30-degree bin, uniform 0.083):")
print(AG.head(4).round(3).to_string(index=False))
chans = pd.DataFrame([dict(session=r["session"], channel=r["theta"]["channel"], location=r["theta"]["channel_location"],
                           ref_deg=np.degrees(r["theta"]["ref"]), peak_bin_share=r["theta"]["spike_phase_hist"].max() / r["theta"]["spike_phase_hist"].sum())
                      for r in R if "theta" in r])
print("sessions with theta:", len(chans), "; ripple channel location:", chans.location.value_counts().to_dict())
print("phase of maximal pyramidal firing (deg, per session): median %.0f, range %.0f to %.0f; share of spikes in the peak 30-degree bin: median %.3f (uniform 0.083)"
      % (chans.ref_deg.median(), chans.ref_deg.min(), chans.ref_deg.max(), chans.peak_bin_share.median()))
for k, X in THS.items():
    print(f"correct trials in the theta table ({k}):", X[(X.correct == 1) & (X.half == "h1") & (X.win == "post")].type.value_counts().to_dict(), "; sessions", X.session.nunique())

## fig3c-theta-phase-segregation

> After the update cue on switch trials, hippocampal non-local goal codes are strongest in the first half of the theta cycle and local codes in the second half.

In [ ]:
def theta_loc_model(central, TH=TH):
    d = TH[(TH.update_type == 2) & (TH.correct == 1) & (TH.win == "post") & TH.half.isin(["h1", "h2"])]
    d = d.rename(columns={central: "centralzone"})
    L = P.long_format(d[["animal", "session", "trial", "half", "init", "new", "centralzone"]], ["init", "new", "centralzone"], name="where")
    L["pred"] = L.half + "|" + L["where"]
    return P.lme(L, "value", "pred"), d
for central, lab in (("central", "(a) whole central arm"), ("home_window", "(b) central window around the update cue")):
    res, d = theta_loc_model(central)
    print(lab)
    report(res, [("h1|init", "h1|centralzone"), ("h1|new", "h1|centralzone"), ("h2|centralzone", "h2|init"), ("h2|centralzone", "h2|new")])
    print("   trial means:", d.groupby("half")[["init", "new", "centralzone"]].mean().round(4).to_dict("index"))
print("expected: initial 0.13/0.13, new 0.13/0.12, central 0.12/0.14 (-pi to 0 / 0 to pi)")
for src, vlab in (("stored", "the phase stored in the file (the paper's code)"), ("computed, rule=train", "computed phase, sessions under the code's 50-trial count")):
  print(f"\nvariant: {vlab}")
  for central in ("central", "home_window"):
    res, d = theta_loc_model(central, THS[src])
    report(res, [("h1|init", "h1|centralzone"), ("h1|new", "h1|centralzone"), ("h2|centralzone", "h2|init"), ("h2|centralzone", "h2|new")], label=central)
    print("   trial means:", d.groupby("half")[["init", "new", "centralzone"]].mean().round(4).to_dict("index"))

**Result: not reproduced.** The central zone is decoded more than either goal arm in both half cycles under both readings (central window: first half central 0.148 against initial 0.129 and new 0.130; second half 0.151 against 0.127 and 0.126; all Tukey p at the precision floor), so the goal arms do not exceed central in the first half. The modulation goes the stated way, central higher and goal arms lower in the second half than in the first, and the goal arm values (0.129, 0.127, 0.130, 0.126) are within tolerance; the central values are not (whole arm 0.66 and 0.67, window 0.148 and 0.151, against 0.12 and 0.14). The phase stored in the file (central window 0.148 and 0.151) and the code's count of trials give the same pattern.

## fig3b-theta-pre-post

> At the theta timescale, goal arm decoding increases and central decoding decreases from before to after the update cue on switch trials.

In [ ]:
for src, central, clab in [("computed", "central", "(a) whole central arm"), ("computed", "home_window", "(b) central window"),
                           ("stored", "central", "variant, stored phase, (a)"), ("computed, rule=train", "central", "variant, rule=train, (a)")]:
    TH = THS[src]
    for half, hlab in (("h1", "first half cycle"), ("all", "whole cycle")):
        d = TH[(TH.update_type == 2) & (TH.correct == 1) & (TH.half == half)].rename(columns={central: "centralzone"})
        L = P.long_format(d[["animal", "session", "trial", "win", "init", "new", "centralzone"]], ["init", "new", "centralzone"], name="where")
        L["pred"] = L.win + "|" + L["where"]
        res = P.lme(L, "value", "pred")
        print(f"{clab}, {hlab}")
        report(res, [("post|init", "pre|init"), ("post|new", "pre|new"), ("post|centralzone", "pre|centralzone")])
        if src == "computed" and central == "central" and half == "h1":
            print("   trial means:", d.groupby("win")[["init", "new", "centralzone"]].mean().round(4).to_dict("index"))
print("expected: initial 0.11 -> 0.13, new 0.11 -> 0.13, central 0.15 -> 0.12")

**Result: reproduced.** Under every reading both goal arms increase from before to after the cue (first half cycle, whole central arm: initial 0.1140 to 0.1291, new 0.1145 to 0.1296, Tukey p = 5.8e-12 and 5.5e-12) and central decoding decreases (whole arm 0.70 to 0.66, central window 0.177 to 0.149 in model means, p at the precision floor). The goal arm values are within tolerance; the central values are not (paper 0.15 to 0.12), as the central zone of the paper is not known. The stored phase gives the same directions (p = 6.0e-11, 1.8e-11 and at the floor), and so does the code's count of trials.

## sfig6-theta-switch-over-delay

> At the theta timescale, goal arm decoding after the update cue is larger on switch than on delay only trials.

In [ ]:
for src, arm in [("computed", "init"), ("computed", "new"), ("stored", "init"), ("stored", "new"), ("computed, rule=train", "init"), ("computed, rule=train", "new")]:
    TH = THS[src]
    print(f"phase {src}:")
    d = TH[(TH.correct == 1) & (TH.win == "post") & TH.half.isin(["h1", "h2"])]
    d = d.assign(pred=d.type + "|" + d.half)
    res = P.lme(d, arm, "pred")
    print(f"{arm} arm: trials {d[d.half == 'h1'].type.value_counts().to_dict()} (paper 1295 switch, 552 stay, 927 delay only)")
    report(res, [("switch|h1", "delay only|h1"), ("switch|h2", "delay only|h2")])

**Result: reproduced.** For both arms the ANOVA (initial F(5) = 31.0, new F(5) = 20.1, p < 1e-19) and the switch against delay only contrasts in both half cycles give p < 1e-8 with switch higher, with either phase and with the code's count of trials (largest 9.25e-9, new arm, second half, code's count), and p < 1e-5 on the small degrees of freedom (41 or 49).

## fig3d-new-arm-theta-modulated

> After the update cue, decoding of the new goal arm differs between theta half cycles while decoding of the initial arm does not.

In [ ]:
for src in ("computed", "stored", "computed, rule=train"):
  TH = THS[src]
  d = TH[(TH.update_type == 2) & (TH.correct == 1) & (TH.win == "post") & TH.half.isin(["h1", "h2"])]
  for arm in ("new", "init", "central"):
    res = P.lme(d, arm, "half")
    print(f"phase {src}, {arm}: F({res['df1']}) = {res['F']:.2f}, p = {res['p_anova']:.3g}; first half {res['emm']['h1']:.4f}, second half {res['emm']['h2']:.4f}")

**Result: reproduced.** On correct switch trials after the cue, the new arm differs between half cycles (0.1286 against 0.1246, p = 0.0049) and the initial arm does not (0.1287 against 0.1266, p = 0.15). With the stored phase, p = 0.015 and p = 0.25; with the code's count of trials, p = 0.0061 and p = 0.22.

## Choice decoding (fig4a and the claims built on it): timing

The choice estimate is a per-session LSTM (S10): 10 units, a sigmoid dense unit, Adam with learning rate 0.1, batch 32, 20 epochs, on the unresampled behaviour (about 110 samples per second) padded to the longest trial, stratified 6-fold cross-validation repeated 3 times, so 18 models per session. Every claim on choice decoding (S11) decodes this estimate. The cell below builds the model as the paper's code does (Keras `Normalization`, `LSTM(10, return_sequences=True)`, `Dense(1, sigmoid)`) on one session's delay only trials and times two epochs of one fold, to estimate the whole.

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(21)
i = int(ASSETS.index[ASSETS.session == "S25-210906"][0])
s = P.load_session(ASSETS.asset_id[i]); trl = P.final_trials(s); dly = trl[trl.update_type == 1]
X, Y = [], []
for _, rr in dly.iterrows():
    i0, i1 = np.searchsorted(s["t"], [rr.start_time, rr.t_choice_made])
    yy = s["xy"][i0:i1, 1]
    X.append(np.c_[s["trans"][i0:i1], s["rot"][i0:i1], yy / yy.max(), s["view_angle"][i0:i1]])
    Y.append(np.full(i1 - i0, float(rr.choice == 2)))
lens = np.array([len(x) for x in X]); keep = lens <= 2 * lens.mean()      # [paper Methods] trials longer than twice the mean excluded
X = [x for x, k in zip(X, keep) if k]; Y = [y for y, k in zip(Y, keep) if k]
Tmax = max(len(x) for x in X)
Xp = tf.keras.preprocessing.sequence.pad_sequences(X, dtype="float64", padding="post", maxlen=Tmax)
Yp = tf.keras.preprocessing.sequence.pad_sequences(Y, dtype="float64", padding="post", maxlen=Tmax)
norm = tf.keras.layers.Normalization(axis=-1); norm.adapt(np.vstack(X))
model = tf.keras.Sequential([tf.keras.Input((Tmax, 4)), norm, tf.keras.layers.LSTM(10, return_sequences=True),
                             tf.keras.layers.Dense(1, activation="sigmoid")])
model.compile(optimizer=tf.keras.optimizers.Adam(0.1), loss="binary_crossentropy")
n5 = int(len(X) * 5 / 6)
t = time.time(); model.fit(Xp[:n5], Yp[:n5], batch_size=32, epochs=1, verbose=0); t_first = time.time() - t
t = time.time(); model.fit(Xp[:n5], Yp[:n5], batch_size=32, epochs=1, verbose=0); t_epoch = time.time() - t
dl = [int((r["trials"].update_type == 1).sum()) for r in R]
per_session = 18 * 20 * t_epoch / 60
whole = sum(per_session * d / len(dly) for d in dl)
print(f"session {ASSETS.session[i]}: {len(X)} delay only trials, longest {Tmax} samples (mean {lens.mean():.0f})")
print(f"one epoch on 5/6 of them: first {t_first:.1f} s, second {t_epoch:.1f} s")
ncpu = os.cpu_count()
print(f"estimate: {per_session:.1f} min for this session's 18 models x 20 epochs; scaled by delay only trial counts, "
      f"{whole / 60:.1f} h for all 66 sessions, against a 60 min limit for the notebook")
print(f"this machine has {ncpu} cores; one session per core would take at best {whole / 60 / ncpu:.1f} h, "
      f"{'over' if whole / ncpu > 60 else 'within'} the 60 min limit")

**Result: the choice-decoding claims are not run.** The estimate printed above for all 66 sessions, in one process and at best on this machine's cores, is beyond the notebook's 60 min limit (the cell prints which), so fig4a-lstm-choice-estimate and every claim that decodes the choice estimate are blocked as `too_large`, as the protocol's rule for an estimate over the notebook limit says. Epoch times vary from run to run, so the hours printed change when the notebook is re-run.

## fig6e-predict-outcome, CA1 half: timing

> The post-update difference between initial and new goal codes predicts whether a switch trial is correct, above shuffled labels, with 61.43% accuracy from prefrontal choice decoding and 55.83% from hippocampal position decoding.

The PFC half decodes the LSTM choice estimate, which is not run (above), and the criterion judges both regions together, so the claim cannot be decided here. The CA1 half needs no LSTM: one feature per correct or incorrect switch trial (initial minus new, 0 to 1.5 s, raw values), the smaller class resampled with replacement to the size of the larger (random_state 123), an RBF SVM with C and gamma from a 10-iteration randomized search under leave-one-out (C from loguniform(1, 1000), gamma from loguniform(1e-4, 1)), accuracy by leave-one-out, and the same on shuffled labels 1000 times (criterion). The cell below builds the CA1 feature table, times one SVM fit for each of 10 (C, gamma) draws on the balanced trials, and estimates the whole procedure: per run 10 x N search fits and N accuracy fits, for the real labels and 1000 shuffles.

In [ ]:
from sklearn.svm import SVC
from sklearn.utils import resample
from scipy.stats import loguniform
T6 = TAB[("CA1", "final")]
d6 = T6[T6.update_type == 2].assign(f=lambda x: x.init - x.new).dropna(subset=["f"])
cor, inc = d6[d6.correct == 1], d6[d6.correct == 0]
big, small = (cor, inc) if len(cor) >= len(inc) else (inc, cor)
B6 = pd.concat([big, resample(small, replace=True, n_samples=len(big), random_state=123)])
X6, y6 = B6[["f"]].to_numpy(), B6.correct.to_numpy()
print(f"CA1 switch trials: {len(cor)} correct, {len(inc)} incorrect (paper 1702 switch trials in all); balanced N = {len(X6)}")
print("feature by outcome (initial minus new, 0 to 1.5 s):", d6.groupby("correct").f.agg(["mean", "std", "count"]).round(4).to_dict("index"))
Cs = loguniform(1, 1000).rvs(10, random_state=123); gs = loguniform(1e-4, 1).rvs(10, random_state=124)
fit_t = []
for c_, g_ in zip(Cs, gs):
    t = time.time(); SVC(C=c_, gamma=g_).fit(X6[:-1], y6[:-1]); fit_t.append(time.time() - t)
n_fit = (10 + 1) * len(X6)
print("seconds per fit for the 10 draws:", np.round(fit_t, 2).tolist())
h_real = n_fit * np.mean(fit_t) / 3600
print(f"estimate: {n_fit} fits per run, {h_real:.1f} h for the real labels alone and "
      f"{1001 * h_real / 24:.0f} days with 1000 shuffles, single process, against a 60 min limit")
print(f"real labels alone: {'over' if h_real > 1 else 'within'} the limit; with 1000 shuffles: "
      f"{'over' if 1001 * h_real > 1 else 'within'} the limit")


**Result: not run.** The CA1 half has 1241 correct and 395 incorrect switch trials, 2482 after balancing, so the paper's procedure (leave-one-out search and accuracy) needs 27302 fits per run. At the fit time printed above, the real labels alone take of the order of an hour, close to the 60 min limit (the cell prints on which side; fit times vary from run to run), and the 1000-shuffle permutation test the criterion asks for takes weeks. The PFC half needs the LSTM choice estimate, and the criterion judges both regions together. The claim stays blocked as `too_large`.

## Summary

| Claim | Outcome |
|---|---|
| recordings | Partially reproduced |
| trial-mix | Not reproduced |
| task-timing | Reproduced |
| fig1b-above-chance | Reproduced |
| fig1b-window-performance | Reproduced |
| fig1d-heading-switch | Reproduced |
| fig5a-stay-performance | Reproduced |
| tuning-tiles-track | Reproduced |
| decoding-validation | Reproduced |
| fig2e-hpc-goal-increase | Reproduced |
| fig2e-hpc-arms-equal | Reproduced |
| sfig2e-goal-bias | Reproduced |
| sfig3c-goals-over-retrospective | Not reproduced |
| sfig3c-retrospective-unchanged | Not reproduced |
| sfig3ab-original-vs-update-cue | Partially reproduced |
| sfig2f-all-trials-encoder | Partially reproduced |
| sfig5f-pfc-wider-spread | Partially reproduced |
| sfig5g-ca1-more-goal-coding | Reproduced |
| fig2g-pfc-no-goal-increase | Not reproduced |
| sfig5e-pfc-windows | Not reproduced |
| fig3c-theta-phase-segregation | Not reproduced |
| fig3b-theta-pre-post | Reproduced |
| sfig6-theta-switch-over-delay | Reproduced |
| fig3d-new-arm-theta-modulated | Reproduced |
| fig5d-stay-hpc-like-delay | Reproduced |
| fig5d-stay-hpc-below-switch | Reproduced |
| sfig10-stay-transient-new-arm | Reproduced |
| fig6b-hpc-new-lower-incorrect | Reproduced |
| fig6b-hpc-initial-no-difference | Reproduced |
| fig6-hpc-pre-cue-initial-higher-incorrect | Reproduced |
| sfig12-delay-hpc-no-outcome-difference | Not reproduced |
| fig7c-hpc-commitment-new-arm | Partially reproduced |
| fig7c-hpc-commitment-initial-arm | Reproduced |
| fig4a-lstm-choice-estimate and 18 claims on choice decoding | Blocked: too large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Decoding is a numpy implementation of pynapple's `decode_1d` (uniform prior). The ANOVA is a Wald test without Kenward-Roger degrees of freedom, and Tukey adjustment uses the studentized range with large degrees of freedom (Kenward-Roger needs R, which is not available here). Every contrast also prints its Tukey p on as many degrees of freedom as there are sessions less the levels, a deliberately small value, to show how much the degrees of freedom could move it. Window means weight each 0.2 s bin equally, as the code's mean over samples does. On the small degrees of freedom, three contrasts that decide outcomes cross 0.05: sfig3c-retrospective-unchanged (0.036 to 0.050), sfig12-delay-hpc-no-outcome-difference (0.043 to 0.053) and the new arm front half of sfig3ab-original-vs-update-cue (0.044 to 0.063); the main-rule tests of fig2g-pfc-no-goal-increase and sfig5e-pfc-windows stay below 0.05. Session inclusion follows the claims file's shared method S1 as a reader amended it in answer to this attempt's question: at least 50 final-task trials of the session, as the Methods say (55 CA1 and 46 PFC sessions, against the paper's 56 and 47). The code's count of the encoder training trials (47 and 37 sessions) is shown for every test; it would change two outcomes, sfig3c-retrospective-unchanged (p = 0.14 against 0.036) and sfig12-delay-hpc-no-outcome-difference (p = 0.096 against 0.043). Update-aligned decoding uses the code's time grid, 0.2 s bins with edges on the cue and centres at the code's sample times (-2.5, ..., -0.1, 0.1, ..., 2.5 s), and windows take bins by centre as the code does. The round-1 grid, with one bin centred on the cue and counted after it, is shown as a variant; it gives other outcomes for three claims (sfig3c-retrospective-unchanged, fig5d-stay-hpc-below-switch and fig7c-hpc-commitment-initial-arm), so those outcomes rest on the 0.1 s by which the two grids differ. It also changes the second test of fig2g-pfc-no-goal-increase under reading b, but reading a fails on both grids, so that outcome is not reproduced either way. The arm halves of sfig3ab follow the code's split; the spread of sfig5f follows the code's rule, with the first-crossing reading as a variant. Theta phase uses a Parks-McClellan band-pass on the LFP decimated to 200 Hz; the phase stored in the file, which the paper's code reads, gives the same outcomes. Position is labelled metres but is in virtual units, and view angle and the stored theta phase are labelled degrees but are in radians. The LSTM choice estimate was not run (timed above), so fig4a-lstm-choice-estimate and the 18 claims on choice decoding are blocked.

What was found about the dataset. The behaviour and trial tables reproduce the paper's counts exactly (7172 delay only, 1840 switch and 717 stay trials; 6252, 1002 and 117 windows of 15 trials). The units of five sessions (S20-210517, S28-211114, S29-211104, S29-211105, S34-220617; 633 units) carry no region or cell type, and the `ch` column cannot map them to a probe; with them, the unit total equals the paper's 6449. Position is labelled metres but is in virtual units (y up to 286), and view angle (-1.05 to 1.05) and the stored theta phase (-pi to pi) are labelled degrees but are in radians. The `turn_type` description `{'right': 1, 'left': 2}` disagrees with the data: on correct delay only trials, `turn_type` 1 trials end at negative x, the arm the paper's code calls left, and choice 1. The Methods' window of 40 trials for Figure 1b does not give the reported window counts; 15 trials, the code's default, does.